# Data Exploration and Cleaning


## 1. Load Data


In [422]:
import pandas as pd

ratings_538 = pd.read_csv("../data/raw/538 Ratings.csv")
ap_polls = pd.read_csv("../data/raw/AP Poll Data.csv")
kenpom = pd.read_csv("../data/raw/KenPom Barttorvik.csv")
team_stats = pd.read_csv("../data/raw/cbb.csv")

**2.Inspect Data Sources**

### 2.1 538 Ratings

In [423]:
ratings_538.columns
#standardising column names
ratings_538.columns = ( ratings_538.columns.str.strip() .str.upper().str.replace(" ", "_"))

In [424]:
#checking for duplacetes
ratings_538.duplicated().sum()

np.int64(0)

In [425]:
#check duplicate team-seasons
ratings_538.duplicated(subset=["TEAM", "YEAR"]).sum()

np.int64(0)

In [426]:
#checking missing values
ratings_538.isna().sum()

YEAR                 0
TEAM_NO              0
TEAM                 0
SEED                 0
ROUND                0
POWER_RATING         0
POWER_RATING_RANK    0
dtype: int64

### 2.2 Ap polls

In [427]:
#standardising column names
ap_polls.columns = (ap_polls.columns.str.strip().str.upper().str.replace(" ", "_"))

### 2.2 AP Poll Data

In [428]:
#checking for duplacates
ap_polls.duplicated().sum()

np.int64(0)

AP Poll Processing:The AP Poll dataset contains multiple observations for each team during a season because rankings are released weekly. To create one observation per team-season for merging, the final available poll for each team is kept.

In [429]:
ap_polls.head()
#only getting last ap poll of the seasson
ap = ap_polls[ap_polls["YEAR"] <= 2024].copy()

ap = ap.sort_values("WEEK")

ap = ap.groupby(["YEAR", "TEAM"],as_index=False).last()

ap = ap[["YEAR", "TEAM", "AP_VOTES", "AP_RANK"]]
ap.head()

,YEAR,TEAM,AP_VOTES,AP_RANK
0,2008,Akron,2.0,41.0
1,2008,Alabama,26.0,36.0
2,2008,American,1.0,43.0
3,2008,Arizona,11.0,33.0
4,2008,Arizona St.,4.0,36.0


In [430]:
#check duplicate team-seasons
ap.duplicated(subset=["TEAM", "YEAR"]).sum()

np.int64(0)

In [431]:
#checking missing values
ap.isna().sum()

YEAR        0
TEAM        0
AP_VOTES    0
AP_RANK     0
dtype: int64

### 2.3 KenPom and Barttorvik

In [432]:
#standardising 
kenpom.columns = (kenpom.columns.str.strip().str.upper().str.replace(" ", "_"))

In [433]:
#checking for duplacates
kenpom.duplicated().sum()

np.int64(0)

In [434]:
#check duplicate team-seasons
kenpom.duplicated(subset=["TEAM", "YEAR"]).sum()

np.int64(0)

In [435]:
#checking missing values
kenpom.isna().sum()

YEAR              0
CONF              0
CONF_ID           0
QUAD_NO           0
QUAD_ID           0
                 ..
FT%_RANK          0
OP_FT%_RANK       0
PPPO_RANK         0
PPPD_RANK         0
ELITE_SOS_RANK    0
Length: 103, dtype: int64

### 2.4 CBB Team Statistics

In [436]:
#standardising 
team_stats.columns = (team_stats.columns.str.strip().str.upper().str.replace(" ", "_"))

In [437]:
#checking for duplacates
team_stats.duplicated().sum()

np.int64(0)

In [438]:
#check duplicate team-seasons
team_stats.duplicated(subset=["TEAM", "YEAR"]).sum()


np.int64(0)

In [439]:
#checking missing values
team_stats.isna().sum()

TEAM             0
CONF             0
G                0
W                0
ADJOE            0
ADJDE            0
BARTHAG          0
EFG_O            0
EFG_D            0
TOR              0
TORD             0
ORB              0
DRB              0
FTR              0
FTRD             0
2P_O             0
2P_D             0
3P_O             0
3P_D             0
ADJ_T            0
WAB              0
POSTSEASON    3433
SEED          3433
YEAR             0
dtype: int64

## 3. Season Coverage

##### checking the range of years for all datasets and then fixing accordingly 

In [440]:
print("538:", ratings_538["YEAR"].min(), "-", ratings_538["YEAR"].max())

538: 2016 - 2024


In [441]:
print("AP:", ap["YEAR"].min(), "-", ap["YEAR"].max())

AP: 2008 - 2024


In [442]:
print("KenPom:", kenpom["YEAR"].min(), "-", kenpom["YEAR"].max())

KenPom: 2008 - 2026


In [443]:
print("CBB:", team_stats["YEAR"].min(), "-", team_stats["YEAR"].max())

CBB: 2013 - 2025


#### range that can fit all data sets is 2016-2024 so fixing datasets accordingly

In [444]:
ratings_538 = ratings_538[ratings_538["YEAR"].between(2016, 2024)]
#checking if range is valid
print("538:", ratings_538["YEAR"].min(), "-", ratings_538["YEAR"].max())


538: 2016 - 2024


In [445]:
ap = ap[ap["YEAR"].between(2016, 2024)]
print("AP:", ap["YEAR"].min(), "-", ap["YEAR"].max())


AP: 2016 - 2024


In [446]:
kenpom = kenpom[kenpom["YEAR"].between(2016, 2024)]
print("KenPom:", kenpom["YEAR"].min(), "-", kenpom["YEAR"].max())

KenPom: 2016 - 2024


In [447]:
team_stats = team_stats[team_stats["YEAR"].between(2016, 2024)]
print("CBB:", team_stats["YEAR"].min(), "-", team_stats["YEAR"].max())

CBB: 2016 - 2024


## 4. Team Name Standardization

### 4.1 basic cleanup

In [448]:
ratings_538["TEAM"] = ratings_538["TEAM"].str.strip()
ap["TEAM"] = ap["TEAM"].str.strip()
kenpom["TEAM"] = kenpom["TEAM"].str.strip()
team_stats["TEAM"] = team_stats["TEAM"].str.strip()

### 4.1 creating sets of team names

In [449]:
teams_538 = set(ratings_538["TEAM"].unique())
teams_ap = set(ap["TEAM"].unique())
teams_kenpom = set(kenpom["TEAM"].unique())
teams_stats = set(team_stats["TEAM"].unique())

### 4.3 Compare 538 vs. team_stats

In [450]:
print("538 only:")
print(sorted(teams_538 - teams_stats))

print("\nTeam Stats only:")
print(len(teams_stats))

538 only:
[]

Team Stats only:
369


### 4.4 Compare Ap vs. team_stats

In [451]:
print("AP only:")
print(sorted(teams_ap - teams_stats))

print("\nTeam Stats only:")
print(sorted(teams_stats - teams_ap))

AP only:
['Boise State', 'Colorado State', 'Florida State', 'Iowa State', 'Loyola Maryland', 'Miami', 'Michigan State', 'Mississippi State', 'Missouri State', 'Murray State', 'New Mexico State', 'Ohio State', 'Oklahoma State', 'Purdue Fort Wayne', 'San Diego State', 'South Dakota State', 'Texas State', 'Utah State', 'Weber State']

Team Stats only:
['Air Force', 'Alabama A&M', 'Alabama St.', 'Albany', 'Alcorn St.', 'American', 'Arkansas Little Rock', 'Arkansas Pine Bluff', 'Austin Peay', 'Bellarmine', 'Bethune Cookman', 'Binghamton', 'Boston University', 'Bowling Green', 'Brown', 'Bryant', 'Bucknell', 'Cal Baptist', 'Cal Poly', 'Cal St. Fullerton', 'Cal St. Northridge', 'Campbell', 'Canisius', 'Central Arkansas', 'Central Connecticut', 'Charleston Southern', 'Charlotte', 'Chicago St.', 'Coastal Carolina', 'Colgate', 'Columbia', 'Coppin St.', 'Dartmouth', 'DePaul', 'Delaware', 'Delaware St.', 'Denver', 'Detroit', 'Dixie St.', 'Drexel', 'East Carolina', 'Eastern Illinois', 'Eastern Kentu

In [452]:
### 4.4 KemPom vs. team_stats

In [453]:
print("KenPom only:")
print(sorted(teams_kenpom - teams_stats))

print("\nTeam Stats only:")
print(sorted(teams_stats - teams_kenpom))

KenPom only:
[]

Team Stats only:
['Air Force', 'Alabama A&M', 'Alabama St.', 'Albany', 'Alcorn St.', 'American', 'Arkansas Little Rock', 'Arkansas Pine Bluff', 'Arkansas St.', 'Army', 'Ball St.', 'Bellarmine', 'Bethune Cookman', 'Binghamton', 'Boston College', 'Boston University', 'Bowling Green', 'Brown', 'Cal Baptist', 'Cal Poly', 'Cal St. Northridge', 'Campbell', 'Canisius', 'Central Arkansas', 'Central Connecticut', 'Central Michigan', 'Charleston Southern', 'Charlotte', 'Chicago St.', 'Coastal Carolina', 'Columbia', 'Coppin St.', 'Cornell', 'Dartmouth', 'DePaul', 'Delaware St.', 'Denver', 'Detroit', 'Dixie St.', 'East Carolina', 'Eastern Illinois', 'Eastern Kentucky', 'Eastern Michigan', 'Elon', 'Evansville', 'FIU', 'Fairfield', 'Florida A&M', 'Fordham', 'Fort Wayne', 'George Mason', 'George Washington', 'Georgia', 'Georgia Southern', 'Harvard', 'High Point', 'Hofstra', 'Houston Baptist', 'Houston Christian', 'IPFW', 'IUPUI', 'Idaho', 'Idaho St.', 'Illinois Chicago', 'Illinois St

### 4.6 Creating name map

##### when looking at the data sets most on the naming differences come from the Ap Poll since the other sets I made were empty

In [454]:

team_name_map = {
    "Boise State": "Boise St.",
    "Colorado State": "Colorado St.",
    "Florida State": "Florida St.",
    "Iowa State": "Iowa St.",
    "Loyola Maryland": "Loyola MD",
    "Miami": "Miami FL",
    "Michigan State": "Michigan St.",
    "Mississippi State": "Mississippi St.",
    "Missouri State": "Missouri St.",
    "Murray State": "Murray St.",
    "New Mexico State": "New Mexico St.",
    "North Carollina": "North Carolina",
    "Ohio State": "Ohio St.",
    "Oklahoma State": "Oklahoma St.",
    "San Diego State": "San Diego St.",
    "South Dakota State": "South Dakota St.",
    "Texas State": "Texas St.",
    "Utah State": "Utah St.",
    "Weber State": "Weber St."
}


### 4.6 applying mapping to data sets to naming is consistent

In [455]:
ratings_538["TEAM"] = ratings_538["TEAM"].replace(team_name_map)
ap["TEAM"] = ap["TEAM"].replace(team_name_map)
kenpom["TEAM"] = kenpom["TEAM"].replace(team_name_map)
team_stats["TEAM"] = team_stats["TEAM"].replace(team_name_map)

### 4.7 Compare team names again

In [456]:
teams_538 = set(ratings_538["TEAM"].unique())
teams_ap = set(ap["TEAM"].unique())
teams_kenpom = set(kenpom["TEAM"].unique())
teams_stats = set(team_stats["TEAM"].unique())

In [457]:
print("538 unmatched:", sorted(teams_538 - teams_stats))
print("AP unmatched:", sorted(teams_ap - teams_stats))
print("KenPom unmatched:", sorted(teams_kenpom - teams_stats))

538 unmatched: []
AP unmatched: ['Purdue Fort Wayne']
KenPom unmatched: []


### 4.9 checking team and year 

In [458]:
keys_538 = set(zip(ratings_538["TEAM"], ratings_538["YEAR"]))
keys_ap = set(zip(ap["TEAM"], ap["YEAR"]))
keys_kenpom = set(zip(kenpom["TEAM"], kenpom["YEAR"]))
keys_stats = set(zip(team_stats["TEAM"], team_stats["YEAR"]))

In [459]:
print("538 unmatched team-seasons:")
print(sorted(keys_538 - keys_stats))

print("\nAP unmatched team-seasons:")
print(sorted(keys_ap - keys_stats))

print("\nKenPom unmatched team-seasons:")
print(sorted(keys_kenpom - keys_stats))

538 unmatched team-seasons:
[('Little Rock', 2016)]

AP unmatched team-seasons:
[('Little Rock', 2016), ('Purdue Fort Wayne', 2017)]

KenPom unmatched team-seasons:
[('Little Rock', 2016)]


### 4.10 making spesific fixed to fix ramaining unmatched

In [460]:
ratings_538.loc[(ratings_538["TEAM"] == "Little Rock") &( ratings_538["YEAR"] == 2016), "TEAM"] = "Arkansas Little Rock"

ap.loc[(ap["TEAM"] == "Little Rock") & (ap["YEAR"] == 2016),"TEAM"] = "Arkansas Little Rock"

kenpom.loc[ (kenpom["TEAM"] == "Little Rock") &  (kenpom["YEAR"] == 2016), "TEAM"] = "Arkansas Little Rock"

ap.loc[(ap["TEAM"] == "Purdue Fort Wayne") & (ap["YEAR"] == 2017), "TEAM"] = "Fort Wayne"

#### checking one more time

In [461]:
keys_538 = set(zip(ratings_538["TEAM"], ratings_538["YEAR"]))
keys_ap = set(zip(ap["TEAM"], ap["YEAR"]))
keys_kenpom = set(zip(kenpom["TEAM"], kenpom["YEAR"]))
keys_stats = set(zip(team_stats["TEAM"], team_stats["YEAR"]))


print("538 unmatched team-seasons:")
print(sorted(keys_538 - keys_stats))

print("\nAP unmatched team-seasons:")
print(sorted(keys_ap - keys_stats))

print("\nKenPom unmatched team-seasons:")
print(sorted(keys_kenpom - keys_stats))

538 unmatched team-seasons:
[]

AP unmatched team-seasons:
[]

KenPom unmatched team-seasons:
[]


### 4.11 Team Name Validation Summary

#### All teams were standardised using the Cbb team statistics data set as the refrence, after fixing any naming inconsisticies and then checking them again against TEAM and YEAR, all relivent teams are matched across datasets.

## 5. Feature Selection

### 5.1 Review Available Features

In [462]:
ratings_538.columns

Index(['YEAR', 'TEAM_NO', 'TEAM', 'SEED', 'ROUND', 'POWER_RATING',
       'POWER_RATING_RANK'],
      dtype='object')

In [463]:
ap.columns

Index(['YEAR', 'TEAM', 'AP_VOTES', 'AP_RANK'], dtype='object')

In [464]:
kenpom.columns

Index(['YEAR', 'CONF', 'CONF_ID', 'QUAD_NO', 'QUAD_ID', 'TEAM_NO', 'TEAM_ID',
       'TEAM', 'SEED', 'ROUND',
       ...
       'BADJT_RANK', 'AVG_HGT_RANK', 'EFF_HGT_RANK', 'EXP_RANK', 'TALENT_RANK',
       'FT%_RANK', 'OP_FT%_RANK', 'PPPO_RANK', 'PPPD_RANK', 'ELITE_SOS_RANK'],
      dtype='object', length=103)

In [465]:
team_stats.columns

Index(['TEAM', 'CONF', 'G', 'W', 'ADJOE', 'ADJDE', 'BARTHAG', 'EFG_O', 'EFG_D',
       'TOR', 'TORD', 'ORB', 'DRB', 'FTR', 'FTRD', '2P_O', '2P_D', '3P_O',
       '3P_D', 'ADJ_T', 'WAB', 'POSTSEASON', 'SEED', 'YEAR'],
      dtype='object')

### 5.2 removing Non-Predictive Columns

In [466]:
ratings_538 = ratings_538[["TEAM", "YEAR", 'POWER_RATING']]
ratings_538.head()

,TEAM,YEAR,POWER_RATING
0,Connecticut,2024,95.6
1,Houston,2024,94.9
2,Purdue,2024,93.5
3,North Carolina,2024,90.2
4,Arizona,2024,91.6


In [467]:
ap =  ap[["YEAR", "TEAM", "AP_RANK"]]
ap.head()

,YEAR,TEAM,AP_RANK
759,2016,Akron,45.0
760,2016,Arizona,17.0
761,2016,Arkansas,53.0
762,2016,Army,42.0
763,2016,BYU,41.0


In [468]:
kenpom = kenpom[["TEAM", "YEAR", "KADJ_O", "KADJ_D","BARTHAG","EFG%",  "EFG%D","TOV%","TOV%D", "OREB%", "DREB%", "FTR", "FTRD"]]
kenpom.head()

,TEAM,YEAR,KADJ_O,KADJ_D,BARTHAG,EFG%,EFG%D,TOV%,TOV%D,OREB%,DREB%,FTR,FTRD
136,Akron,2024,107.009,102.0720,0.596,52.0,48.6,17.2,16.6,29.5,74.4,33.6,29.0
137,Alabama,2024,125.601,103.1180,0.913,56.3,49.9,16.0,15.6,34.9,70.1,35.2,39.6
138,Arizona,2024,121.125,94.5013,0.953,55.0,48.7,16.1,18.1,36.3,76.9,36.7,25.7
139,Auburn,2024,120.579,91.6765,0.957,54.1,43.4,14.9,18.2,32.9,69.7,38.2,41.0
140,Baylor,2024,122.490,100.1850,0.910,55.6,51.3,17.7,17.1,35.1,72.9,40.5,30.9


In [469]:
team_stats = team_stats[["TEAM","YEAR","G","W","SEED","2P_O","2P_D","3P_O","3P_D","ADJ_T","WAB"]]
team_stats.head()

,TEAM,YEAR,G,W,SEED,2P_O,2P_D,3P_O,3P_D,ADJ_T,WAB
0,North Carolina,2016,40,33,1.0,53.9,44.6,32.7,36.2,71.7,8.6
2,Michigan,2018,40,33,3.0,54.7,46.8,35.2,33.2,65.9,6.9
3,Texas Tech,2019,38,31,3.0,52.8,41.9,36.5,29.7,67.5,7.0
4,Gonzaga,2017,39,37,1.0,56.3,40.0,38.2,29.0,71.5,7.7
8,Virginia,2019,38,35,1.0,52.5,45.7,39.5,28.9,60.7,11.1


In [470]:
print(team_stats.columns.tolist())

['TEAM', 'YEAR', 'G', 'W', 'SEED', '2P_O', '2P_D', '3P_O', '3P_D', 'ADJ_T', 'WAB']


### 5.3 Feature Selection Summary

##### for 583 rankins I chose to only keep Team, Year, and Power rating since other peices of data were redundant, for the Ap poll rankings I decided to only keep Team, Year, and The Final Ap ranking that I got from 2.2 since the other data was not important to the model. For Kenpom I decided to keep teh chosen columns since they were the ones that I think can have a stron efficency and adds poll/ranking information and they weren't in the team stats data set. Lastly I chose the ones in the team stats data set since I think they are the varibles with the base team information that are essental for the model I am going to be running.

## 6 Merge data sets

### 6.1 Team stats 

In [471]:
team_seasons = team_stats.copy()
print(team_seasons.shape)

(2836, 11)


### 6.2 Merge kenpom

In [472]:
team_seasons = team_seasons.merge(kenpom, on=["TEAM", "YEAR"],how="left",validate="one_to_one")

print(team_seasons.shape)
team_seasons.head()
print(team_seasons.columns.tolist())

(2836, 22)
['TEAM', 'YEAR', 'G', 'W', 'SEED', '2P_O', '2P_D', '3P_O', '3P_D', 'ADJ_T', 'WAB', 'KADJ_O', 'KADJ_D', 'BARTHAG', 'EFG%', 'EFG%D', 'TOV%', 'TOV%D', 'OREB%', 'DREB%', 'FTR', 'FTRD']


### 6.3 Merge 538

In [473]:
team_seasons = team_seasons.merge(ratings_538,on=["TEAM", "YEAR"],how="left",validate="one_to_one")
print(team_seasons.shape)

(2836, 23)


### 6.4 Merge Ap poll

In [474]:
team_seasons = team_seasons.merge(ap,on=["TEAM", "YEAR"],how="left",validate="one_to_one")

## 7 validate merged data set

### 7.1 check shape

In [475]:
print("Final dataset shape:", team_seasons.shape)
print("Number of rows:", team_seasons.shape[0])
print("Number of columns:", team_seasons.shape[1])

Final dataset shape: (2836, 24)
Number of rows: 2836
Number of columns: 24


In [476]:
print("Original CBB rows:", len(team_stats))
print("Merged dataset rows:", len(team_seasons))

Original CBB rows: 2836
Merged dataset rows: 2836


In [477]:
print("Row counts match:", len(team_stats) == len(team_seasons))

Row counts match: True


### 7.2 Check Duplicate Team-Seasons

In [478]:
duplicates = team_seasons.duplicated(subset=["TEAM", "YEAR"]).sum()

print("Duplicate TEAM/YEAR observations:", duplicates)

Duplicate TEAM/YEAR observations: 0


## 7.3 check missing values

In [479]:
missing_values = team_seasons.isna().sum()

print(missing_values)

TEAM               0
YEAR               0
G                  0
W                  0
SEED            2292
2P_O               0
2P_D               0
3P_O               0
3P_D               0
ADJ_T              0
WAB                0
KADJ_O          2292
KADJ_D          2292
BARTHAG         2292
EFG%            2292
EFG%D           2292
TOV%            2292
TOV%D           2292
OREB%           2292
DREB%           2292
FTR             2292
FTRD            2292
POWER_RATING    2292
AP_RANK         2068
dtype: int64


In [480]:
missing_percent = (team_seasons.isna().mean() * 100)

print(missing_percent[missing_percent > 0])

SEED            80.818054
KADJ_O          80.818054
KADJ_D          80.818054
BARTHAG         80.818054
EFG%            80.818054
EFG%D           80.818054
TOV%            80.818054
TOV%D           80.818054
OREB%           80.818054
DREB%           80.818054
FTR             80.818054
FTRD            80.818054
POWER_RATING    80.818054
AP_RANK         72.919605
dtype: float64


In [481]:
print("CBB years:", sorted(team_stats["YEAR"].unique()))
print("KenPom years:", sorted(kenpom["YEAR"].unique()))

CBB years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
KenPom years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## 7.3 Check Merge Coverage

In [482]:
print("538 coverage:",round(team_seasons["POWER_RATING"].notna().mean() * 100, 2), "%")

print("AP coverage:",round(team_seasons["AP_RANK"].notna().mean() * 100, 2), "%")

print("KenPom coverage:",round(team_seasons["KADJ_O"].notna().mean() * 100, 2), "%")

538 coverage: 19.18 %
AP coverage: 27.08 %
KenPom coverage: 19.18 %


## 7.34 Final Range check

In [483]:
print("Minimum year:", team_seasons["YEAR"].min())
print("Maximum year:", team_seasons["YEAR"].max())

Minimum year: 2016
Maximum year: 2024


In [484]:
print(sorted(team_seasons["YEAR"].unique()))

[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [485]:
print("Rows after 2024:",(team_seasons["YEAR"] > 2024).sum())

Rows after 2024: 0


## Save final data set

In [486]:
team_seasons.to_csv("../data/processed/team_seasons.csv",index=False)